In [3]:
import pandas as pd
import numpy as np
import random 

In [4]:
from typing import Callable

def mse(y_true: pd.Series, y_pred: pd.Series):
    squared_differences = np.square(y_true - y_pred)
    mse = np.mean(squared_differences)
    return mse

def mae(y_true: pd.Series, y_pred: pd.Series):
    absolute_differrences = np.abs(y_true - y_pred)
    mae = np.mean(absolute_differrences)
    return mae

def rmse(y_true: pd.Series, y_pred: pd.Series):
    squared_differences = np.square(y_true - y_pred)
    mse = np.mean(squared_differences)
    rmse = np.sqrt(mse)
    return rmse

def r2(y_true: pd.Series, y_pred: pd.Series):
    y_mean = np.mean(y_true)
    squared_differences = np.sum(np.square(y_true - y_pred))
    squared_differences_mean = np.sum(np.square(y_true - y_mean))
    r2 = 1 - squared_differences / squared_differences_mean
    return r2


def mape(y_true: pd.Series, y_pred: pd.Series):
    mape = 100 * np.mean(np.abs(1 - y_pred / y_true))
    return mape
    

def get_mse(y: pd.Series):
    return ((y - y.mean()) ** 2).mean()

def accuracy(y_true, y_pred) -> float:
    return np.mean(y_true == y_pred)

def precision(y_true, y_pred) -> float:
    mask = y_pred == 1
    return y_true[mask].mean()

def recall(y_true, y_pred) -> float:
    mask = y_true == 1
    return y_pred[mask].mean()

def f1(y_true, y_pred) -> float:
    precision_ = precision(y_true, y_pred)
    recall_ = recall(y_true, y_pred)
    f1 = 2 * precision_ * recall_ / (precision_ + recall_)
    return f1

def roc_auc(y_true: pd.Series, y_pred: pd.Series):
    y_true = y_true.copy()
    y_true.name = 'true'
    y_pred = y_pred.copy()
    y_pred.name = 'pred'
    y_pred = np.round(y_pred, 10)
    
    y = pd.concat([y_pred, y_true], axis=1).sort_values(["pred", "true"], ascending=False)    

    roc_auc = 0 
    for i in range(len(y)):
        if y['true'].iloc[i] == 0:
            p = y['pred'].iloc[i]
            for j in range(i):
                if y['true'].iloc[j] == 1 and y['pred'].iloc[j] > p :
                    roc_auc += 1
                elif y['true'].iloc[j] == 1 and y['pred'].iloc[j] == p:
                    roc_auc += 0.5
    roc_auc /= np.sum(y['true']) * (len(y['true']) - np.sum(y['true']))

    return roc_auc

 
class Leaf:
    def __init__(self, parent, samples, y, depth, samples_indexes):
        self.parent = parent
        self.samples = samples
        self.y = y
        self.depth = depth
        self.samples_indexes = samples_indexes

    def __str__(self):
        return f'{"  " * self.depth} {self.y}'

    def set_argmin_y(self, y, p, reg):
        y = y.loc[self.samples_indexes]
        p = p.loc[self.samples_indexes]
        
        self.y = (y - p).sum() / (p * (1 - p)).sum() + reg
            

class Node:
    def __init__(self, parent=None):
        self.parent = parent
        self.left = None
        self.right = None
        self.samples = None
        self.depth = 1
        self.split_value = None
        self.column = None
        self.ig = None
        self.is_left = None

    def __str__(self):
        return  '\n'.join([
        f'{"  " * self.depth} {self.column} {self.split_value}',
        f'{self.left}',
        f'{self.right}'])

    def predict(self, row: pd.Series):
        if row.loc[self.column] <= self.split_value:
            if isinstance(self.left, Leaf):
                return self.left.y
            else: 
                return self.left.predict(row)
        else:
            if isinstance(self.right, Leaf):
                return self.right.y
            else: 
                return self.right.predict(row)

    def set_argmin_y(self, y, p, reg):
        self.left.set_argmin_y(y, p, reg)
        self.right.set_argmin_y(y, p, reg)

class MyTreeReg:
    def __init__(self, max_depth: int = 5, min_samples_split: int = 2,
                 max_leafs: int = 20, bins=None, N=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.nodes_cnt = 1
        self.leafs_cnt = None
        self.bins = bins
        self.split_values = None
        self.fi = dict()
        self.N = N

    def __str__(self):
        return f"MyTreeReg class: max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, max_leafs={self.max_leafs}"

    def __build_split_values(self, X):
        self.split_values = dict()
        for column in X.columns:
            unique_values = X[column].unique()
            if (len(unique_values) - 1 <= self.bins - 1):
                sorted_unique_values = np.sort(unique_values)
                self.split_values[column] = (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
            else:
                count_elements, split_values = np.histogram(X[column], bins = self.bins)
                self.split_values[column] = split_values[1:-1]

    def predict(self, X: pd.DataFrame):
        y = []
        for index, row in X.iterrows():
            y_index = self.root.predict(row)
            y.append(y_index)
        return pd.Series(y, X.index)

    def get_split_values(self, X, column):
        if self.bins is None:
            unique_values = X[column].unique()
            sorted_unique_values = np.sort(unique_values)
            return (sorted_unique_values[:-1] + sorted_unique_values[1:]) / 2
        else:
            max_value = X[column].max()
            min_value = X[column].min()
            return self.split_values[column][ (min_value <= self.split_values[column]) & (max_value > self.split_values[column])]

    def __init_fi(self, X):
        for column in X.columns:
            self.fi[column] = 0 
    
    def fit(self, X: pd.DataFrame, y: pd.Series):
        self.root = Node()
        self.__init_fi(X)
        self.N = len(y) if self.N is None else self.N
        if self.bins is not None:
            self.__build_split_values(X)
            
        self.build_tree_recursive(self.root, X, y)
        self.leafs_cnt = self.nodes_cnt + 1

    def get_best_split(self, X: pd. DataFrame, y : pd.Series):

        igs = []
        
        for column in X.columns:
            split_values = self.get_split_values(X, column)
            criterion_function = get_mse
            criterion = criterion_function(y)
            
            for split_value in split_values: 

                left_indexes = X.index[X[column] <= split_value]
                left_y = y.loc[left_indexes]
                left_criterion = criterion_function(left_y)
                
                right_indexes = X.index[X[column] > split_value]
                right_y = y.loc[right_indexes]
                right_criterion = criterion_function(right_y)
    
                ig = criterion - left_criterion * len(left_indexes) / len(y) - right_criterion * len(right_indexes) / len(y) 
    
                igs.append((column, split_value, ig))
        if igs:    
            return max(igs, key= lambda x: x[2])
        else:
            return None, None, None

        
    def build_tree_recursive(self, node: Node, X, y):

        node.column, node.split_value, node.ig = self.get_best_split(X, y)
        
        if node.column is not None:
            self.fi[node.column] += len(y) / self.N * node.ig
            
            left_indexes = X.index[X[node.column] <= node.split_value]
            left_cnt = len(left_indexes)
            left_y = y.loc[left_indexes]
            
            if len(left_y.unique()) == 1 or left_cnt < self.min_samples_split or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth \
                :
                node.left = Leaf(node, len(left_y), left_y.mean(), node.depth + 1, left_indexes)
            else:
                node.left = Node(node)
                node.left.depth = node.depth + 1
                node.left.is_left = True
                self.nodes_cnt += 1
                self.build_tree_recursive(node.left, X.loc[left_indexes], left_y)
    
            right_indexes = X.index[X[node.column] > node.split_value]
            right_cnt = len(right_indexes)
            right_y = y.loc[right_indexes]
            
            if len(right_y.unique()) == 1  or right_cnt < self.min_samples_split  or \
                self.nodes_cnt + 1 >= self.max_leafs or node.depth == self.max_depth:
                node.right = Leaf(node, len(right_y), right_y.mean(), node.depth + 1, right_indexes)
            else:
                node.right = Node(node)
                node.right.depth = node.depth + 1
                node.right.is_left = False
                self.nodes_cnt += 1
                self.build_tree_recursive(node.right, X.loc[right_indexes], right_y)
        else:
            parent = node.parent
            if node.is_left:
                parent.left = Leaf(parent, len(y), y.mean(), parent.depth + 1, y.index)
            else:
                parent.right = Leaf(parent, len(y), y.mean(), parent.depth + 1, y.index)
            self.nodes_cnt -= 1

    def set_argmin_y(self, y, p, reg=0):
        self.root.set_argmin_y(y, p, reg)


class MyBoostClf:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=5, min_samples_split=2, 
                 max_leafs=20, bins=16, metric=None, max_features=0.5, max_samples=0.5, random_state=42, reg=0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_leafs = max_leafs
        self.bins = bins
        self.metric = metric
        self.max_features = max_features
        self.max_samples = max_samples
        self.random_state = random_state
        self.reg = reg
        
        self.leafs_cnt = 0
        self.pred_0 = None
        self.trees = []
        self.eps = 1e-15
        self.fi = dict()
        self.scores = []

    def __str__(self):
        return f"MyBoostClf class: n_estimators={self.n_estimators}, learning_rate={self.learning_rate}, "\
              f"max_depth={self.max_depth}, min_samples_split={self.min_samples_split}, "\
              f"max_leafs={self.max_leafs}, bins={self.bins}"

    def log_loss(self, y, p):
        return  - np.mean(y * np.log(p + self.eps)  + (1 - y) * np.log(1 - p + self.eps))
    
    def fit(self, X: pd.DataFrame, y: pd.Series, X_eval, y_eval, early_stopping, verbose=None):
        random.seed(self.random_state)

        for column in X.columns:
            self.fi[column] = 0.0
            
        p = y.mean()
        self.pred_0 = np.log(p / (1 - p))

        antigradient = y - p

        p = pd.Series(p, y.index)
        F = pd.Series(self.pred_0, y.index)

        if self.metric == "accuracy":
            metric = accuracy
        if self.metric == "precision":
            metric = precision
        if self.metric == "recall":
            metric = recall
        if self.metric == "f1":
            metric = f1
        if self.metric == "roc_auc":
            metric = roc_auc
        if self.metric is None:
            metric = self.log_loss

        for i in range(1, self.n_estimators + 1):
            my_tree_reg = MyTreeReg(self.max_depth, self.min_samples_split, self.max_leafs, self.bins, N=len(X))

            cols_smpl_cnt = round(X.shape[1] * self.max_features)
            cols_idx = random.sample(X.columns.to_list(), cols_smpl_cnt)
    
            rows_smpl_cnt = round(X.shape[0] * self.max_samples)
            rows_idx = random.sample(X.index.to_list(), rows_smpl_cnt)
            
            my_tree_reg.fit(X[cols_idx].loc[rows_idx], antigradient.loc[rows_idx])
            
            my_tree_reg.set_argmin_y(y, p, self.reg * self.leafs_cnt)
            self.leafs_cnt += my_tree_reg.leafs_cnt
            self.trees.append(my_tree_reg)

            gamma = my_tree_reg.predict(X)
            if isinstance(self.learning_rate, Callable):    
                F +=  self.learning_rate(i) * gamma
            else:
                F +=  self.learning_rate * gamma
                
            p = np.exp(F) / (1 + np.exp(F))
            antigradient = y - p

            if verbose and i % verbose == 0:
                print(f"{i // verbose}. {self.log_loss(y, p)}")

            
            if early_stopping is not None:
                if self.metric == "roc_auc" or  self.metric is None:
                    y_pred = self.predict_proba(X_eval)
                else:
                    y_pred = self.predict(X_eval)
                    
                self.scores.append(metric(y_eval, y_pred))
                            
                if i > early_stopping:
                    last_early_stoping_score = self.scores[- early_stopping - 1]
                    if self.metric is None and  min(self.scores[-early_stopping:]) > last_early_stoping_score or \
                       self.metric and  max(self.scores[-early_stopping:]) < last_early_stoping_score:
                        self.trees = self.trees[:-early_stopping]
                        self.scores = self.scores[: -early_stopping]
                        break


        if self.scores:
            self.best_score = self.scores[-1]
        else:
            y_pred = self.predict_proba(X)
            self.best_score = metric(y, y_pred)        
            
        for column in self.fi:
            for tree in self.trees:
                self.fi[column] += tree.fi.get(column, 0)

    def predict_proba(self, X: pd.DataFrame):
        F = pd.Series(self.pred_0, X.index)

        for i, tree in enumerate(self.trees):
            if isinstance(self.learning_rate, Callable):    
                F +=  self.learning_rate(i+1) * tree.predict(X)
            else:
                F +=  self.learning_rate * tree.predict(X)

        p = np.exp(F) / (1 + np.exp(F))

        return p

    def predict(self, X):
        p = self.predict_proba(X)
        return p > 0.5
        